# Lab 1: 제조 지식 문서를 ChromaDB에 인덱싱

이 실습에서는 제조 현장 문서를 벡터 데이터베이스(ChromaDB)에 저장하여
의미 기반 검색(Semantic Search)이 가능하도록 인덱싱합니다.

**학습 목표:**
- ChromaDB 클라이언트 생성 및 컬렉션 관리
- Sentence Transformer를 사용한 한국어 임베딩
- 제조 도메인 문서의 벡터 저장 및 검색

In [ ]:
from chromadb import Client
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
import chromadb

# ChromaDB 클라이언트 (로컬)
client = chromadb.Client()
collection = client.create_collection("manufacturing_docs")

# 샘플 제조 문서 (KAMP 스타일)
documents = [
    {"id": "doc1", "text": "CNC 머신 주축 베어링 이상 진동 발생 시 즉시 가동 중단 후 점검 필요. 진동값 3mm/s 초과 시 위험 신호.", "metadata": {"source": "설비매뉴얼", "topic": "베어링"}},
    {"id": "doc2", "text": "용접부 품질 기준: 인장강도 400MPa 이상, 경도 HRC 20~30. 초음파 검사 주기: 월 1회.", "metadata": {"source": "품질기준서", "topic": "용접"}},
    {"id": "doc3", "text": "스마트공장 ROI 사례: 생산성 33.6% 향상, 품질 불량률 44.4% 감소. 초기 투자 2억 → 3년 회수.", "metadata": {"source": "KAMP사례집", "topic": "ROI"}},
    {"id": "doc4", "text": "AutoEncoder 기반 이상탐지: 정상 데이터만 학습, 재구성 오차가 임계값 초과 시 이상 판정. 민감도 조정 가능.", "metadata": {"source": "AI가이드", "topic": "이상탐지"}},
]

# 임베딩 모델 (한국어 지원 다국어 모델)
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
embeddings = model.encode([d["text"] for d in documents]).tolist()

# ChromaDB에 저장
collection.add(
    ids=[d["id"] for d in documents],
    embeddings=embeddings,
    documents=[d["text"] for d in documents],
    metadatas=[d["metadata"] for d in documents]
)
print(f"✅ {len(documents)}개 문서 인덱싱 완료")
print(f"컬렉션 크기: {collection.count()}")

In [ ]:
# 쿼리 테스트
query = "베어링 이상 진동 기준이 뭐야?"
query_embedding = model.encode([query]).tolist()

results = collection.query(
    query_embeddings=query_embedding,
    n_results=2
)
print(f"쿼리: {query}")
print("\n=== 검색 결과 ===")
for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
    print(f"[{i+1}] 출처: {meta['source']} | 주제: {meta['topic']}")
    print(f"    내용: {doc[:100]}...")

## 정리

이것이 RAG의 첫 단계입니다 — 질문과 관련된 문서를 자동으로 찾습니다.

**핵심 흐름:**
```
문서 텍스트 → 임베딩 모델 → 벡터 → ChromaDB 저장
질문 텍스트 → 임베딩 모델 → 벡터 → 유사도 검색 → 관련 문서 반환
```

**다음 Lab:** 검색된 문서를 LLM에 넘겨 최종 답변을 생성하는 파이프라인을 구축합니다.